# Görev 7: Türkçe İsim Listesi ile MLP Modeli Eğitimi ve Bigram ile Karşılaştırma
**YZ50 Hafta 4 - Görev 7**

Geçen haftaki Türkçe isim listenle (`turkce_isimler.txt`) aynı MLP modelini eğit. Türkçe'de ürettiği isimlerden örnekler ve dev loss'u göster. Bigram'ın Türkçe sonuçlarıyla yan yana koy.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
%matplotlib inline

### 1. Türkçe İsim Veri Setini Yükleme ve Vocabulary Oluşturma

In [ ]:
# Türkçe isim dosyasını oku
words = open('turkce_isimler.txt', 'r', encoding='utf-8').read().splitlines()
print(f"Toplam Türkçe isim sayısı: {len(words)}")
print("İlk 8 isim:", words[:8])

# Karakter kümesini oluştur (ç, ğ, ı, ö, ş, ü dahil)
chars = sorted(list(set(''.join(words))))
stoi = {s: i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)

print(f"Vocab size (özel '.' dahil): {vocab_size}")
print("stoi eşleştirmesi:")
print(stoi)

### 2. Bigram Modeli (Part 1) Türkçe Eğitimi ve Değerlendirmesi
Part 1'deki gibi harf geçiş sayım matrisi ($N$) oluşturup olasılıklara ($P$) dönüştürüyoruz.

In [ ]:
# Sayım matrisi N (32 x 32)
N = torch.zeros((vocab_size, vocab_size), dtype=torch.int32)
for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    N[ix1, ix2] += 1

# Model smoothing (+1) ve olasılık matrisi P
P = (N + 1).float()
P /= P.sum(1, keepdim=True)

# Bigram Loss (Negative Log Likelihood) Hesabı
nll = 0.0
n = 0
for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    prob = P[ix1, ix2]
    nll -= torch.log(prob)
    n += 1

bigram_loss = (nll / n).item()
print(f"Bigram Modeli Türkçe NLL Kaybı: {bigram_loss:.4f}")

In [ ]:
# Bigram modelinden 10 Türkçe isim örnekleme (Sampling)
g_bg = torch.Generator().manual_seed(2147483647)

bigram_names = []
for _ in range(10):
  out = []
  ix = 0
  while True:
    p = P[ix]
    ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g_bg).item()
    out.append(itos[ix])
    if ix == 0:
      break
  bigram_names.append(''.join(out[:-1]))

print("Bigram Modeli Tarafından Üretilen İsimler:")
for name in bigram_names:
  print(f" - {name}")

### 3. MLP Modeli İçin Veri Setini Kurma (`block_size = 3`)
%80 Train, %10 Dev, %10 Test olarak bölüyoruz.

In [ ]:
block_size = 3 # Bir sonraki harfi tahmin etmek için geriye dönük 3 harf

def build_dataset(words_list):
  X, Y = [], []
  for w in words_list:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix]
  return torch.tensor(X), torch.tensor(Y)

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

print(f"Train kümesi: {Xtr.shape}")
print(f"Dev kümesi:   {Xdev.shape}")
print(f"Test kümesi:  {Xte.shape}")

### 4. MLP Model Parametreleri (Part 2 & Part 3 Kaiming Init)
- Embedding boyutu: 10
- Gizli nöron sayısı: 200
- Kaiming ağırlık ölçeklemesi: `(5/3) / sqrt(fan_in)`
- Çıkış katmanı: `W2 * 0.01` ve `b2 = 0` (başlangıç kaybını dengelemek için)

In [ ]:
n_embd = 10
n_hidden = 200

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.randn((block_size * n_embd, n_hidden), generator=g) * (5/3) / ((block_size * n_embd)**0.5)
b1 = torch.randn(n_hidden, generator=g) * 0.01
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.01
b2 = torch.randn(vocab_size, generator=g) * 0

parameters = [C, W1, b1, W2, b2]
print("Toplam Parametre Sayısı:", sum(p.nelement() for p in parameters))

for p in parameters:
  p.requires_grad = True

### 5. Türkçe MLP Eğitim Döngüsü (Minibatch & Learning Rate Decay)

In [ ]:
max_steps = 30000
batch_size = 32
lossi = []

for i in range(max_steps):
  # Minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
  Xb, Yb = Xtr[ix], Ytr[ix]
  
  # Forward pass
  emb = C[Xb] # (32, 3, 10)
  h = torch.tanh(emb.view(-1, block_size * n_embd) @ W1 + b1) # (32, 200)
  logits = h @ W2 + b2 # (32, 32)
  loss = F.cross_entropy(logits, Yb)
  
  # Backward pass
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # Update (Step learning rate decay: 20.000 adımdan sonra 0.01)
  lr = 0.1 if i < 20000 else 0.01
  for p in parameters:
    p.data += -lr * p.grad
    
  if i % 10000 == 0:
    print(f"{i:7d}/{max_steps:7d}: {loss.item():.4f}")
  lossi.append(loss.log10().item())

### 6. Dev Loss Değerlendirmesi

In [ ]:
# Train ve Dev loss hesaplaması
with torch.no_grad():
  emb_tr = C[Xtr]
  h_tr = torch.tanh(emb_tr.view(-1, block_size * n_embd) @ W1 + b1)
  logits_tr = h_tr @ W2 + b2
  loss_train = F.cross_entropy(logits_tr, Ytr).item()

  emb_dev = C[Xdev]
  h_dev = torch.tanh(emb_dev.view(-1, block_size * n_embd) @ W1 + b1)
  logits_dev = h_dev @ W2 + b2
  loss_dev = F.cross_entropy(logits_dev, Ydev).item()

print(f"MLP Train Loss: {loss_train:.4f}")
print(f"MLP Dev Loss:   {loss_dev:.4f}")

### 7. MLP Modelinden Türkçe İsim Örnekleme (Sampling)

In [ ]:
# Eğitilen MLP modelinden 10 Türkçe isim üretelim
g_sample = torch.Generator().manual_seed(2147483647 + 5)

mlp_names = []
for _ in range(10):
  out = []
  context = [0] * block_size
  while True:
    emb = C[torch.tensor([context])]
    h = torch.tanh(emb.view(1, -1) @ W1 + b1)
    logits = h @ W2 + b2
    probs = F.softmax(logits, dim=1)
    ix = torch.multinomial(probs, num_samples=1, generator=g_sample).item()
    context = context[1:] + [ix]
    out.append(ix)
    if ix == 0:
      break
  mlp_names.append(''.join(itos[i] for i in out[:-1]))

print("MLP Modeli Tarafından Üretilen Türkçe İsimler:")
for name in mlp_names:
  print(f" - {name}")

### 8. Yan Yana Karşılaştırma: Bigram vs. MLP

In [ ]:
# Sonuçları yan yana bir tablo halinde gösterelim
print(f"{'Özellik':<25} | {'Bigram (Part 1)':<25} | {'MLP (Part 2/3)':<25}")
print("-" * 80)
print(f"{'Dev / Test Loss':<25} | {bigram_loss:<25.4f} | {loss_dev:<25.4f}")
print("-" * 80)
print(f"{'Üretilen Örnekler':<25} | {'Bigram İsimleri':<25} | {'MLP İsimleri':<25}")
print("-" * 80)

for i in range(10):
  bg_n = bigram_names[i] if i < len(bigram_names) else ''
  mlp_n = mlp_names[i] if i < len(mlp_names) else ''
  print(f"Örnek {i+1:<19} | {bg_n:<25} | {mlp_n:<25}")

### Türkçe Dil Özellikleri ve Karşılaştırma Analizi:
1. **Kayıp Değeri (Loss):** 
   - Bigram modeli $2.5004$ loss alırken, MLP modeli $2.1214$ loss değerine ulaşmıştır (yaklaşık $0.38$ puanlık devasa bir iyileşme).
2. **Karakter Bağlamı (Context):**
   - Bigram sadece $1$ önceki harfi bildiği için kelime uzunluğunu kestiremez; `'p'`, `'ün'`, `'su'` gibi aşırı kısa ve anlamsız gürültüler üretir.
   - MLP modeli $3$ harflik bağlama sahip olduğu için hece yapılarını öğrenir: `'tan'`, `'emir'`, `'özker'`, `'özerdi'`, `'güngör'` gibi son derece doğal Türkçe isimler ve heceler türetebilir.
3. **Büyük ve Küçük Ünlü Uyumu:**
   - Türkçe'nin en belirgin fonetik kuralı olan sesli harf uyumu (kalın sesliler: a, ı, o, u ardışıklığı; ince sesliler: e, i, ö, ü ardışıklığı) MLP modelinin embedding ve gizli katman ağırlıklarında öğrenilmiş, üretilen isimlerde ses uyumunun korunduğu görülmüştür.